In [8]:
!pip install reportlab

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ---------------- ----------------------- 0.8/2.0 MB 2.6 MB/s eta 0:00:01
   -------------------------- ------------- 1.3/2.0 MB 2.0 MB/s eta 0:00:01
   -------------------------------- ------- 1.6/2.0 MB 2.1 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 2.1 MB/s  0:00:00


In [21]:
import requests
from bs4 import BeautifulSoup
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import A4

def scrapping2pdf(link, titulo, descripcion):
    nombre_pdf = link.rstrip("/").split("/")[-1]

    # Configuración de PDF
    doc = SimpleDocTemplate(
        f"../doc_pdf/{nombre_pdf}.pdf",
        pagesize=A4,
        rightMargin=40, leftMargin=40,
        topMargin=40, bottomMargin=40
    )
    styles = getSampleStyleSheet()
    story = []

    # Portada
    story.append(Paragraph(f"<para align='center'><b>{titulo}</b></para>", styles["Title"]))
    story.append(Spacer(1, 12))
    story.append(Paragraph(f"<para align='center'>{descripcion}</para>", styles["Normal"]))
    story.append(Paragraph(f"<para align='center'>Libro recopilado de {link}</para>", styles["Normal"]))
    story.append(PageBreak())

    # Obtener lista de capítulos
    response = requests.get(link)
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")

    nombre_pdf = link.rstrip("/").split("/")[-1]
    chapter_links = []
    for a in soup.select("a[href]"):
        href = a["href"]
        if href.startswith(f"/{nombre_pdf}/") and href != f"/{nombre_pdf}/":
            chapter_links.append("https://basecamp.com" + href)

    chapter_links = sorted(set(chapter_links))

    # Índice
    story.append(Paragraph("<b>Index</b>", styles["Heading2"]))
    for i, ch_link in enumerate(chapter_links, 1):
        res = requests.get(ch_link)
        res.encoding = "utf-8"
        chapter_soup = BeautifulSoup(res.text, "html.parser")
        title = chapter_soup.find("h1").get_text(strip=True)
        story.append(Paragraph(f"Chapter {i}. {title}", styles["Normal"]))
    story.append(PageBreak())

    # Capítulos
    for i, ch_link in enumerate(chapter_links, 1):
        res = requests.get(ch_link)
        res.encoding = "utf-8"
        chapter_soup = BeautifulSoup(res.text, "html.parser")

        # Título del capítulo
        title = chapter_soup.find("h1").get_text(strip=True)

        content_elem = chapter_soup.select_one("div.content")
        if not content_elem:
            continue

        # Limpiar footers/publicidad
        for unwanted in content_elem.find_all(["footer"], recursive=True):
            unwanted.decompose()
        for p in content_elem.find_all("p"):
            if "We made" in p.get_text() or "Copyright" in p.get_text():
                p.decompose()
        for a in content_elem.find_all("a"):
            if not a.get_text(strip=True):
                a.string = a["href"]

        # Agregar capítulo
        story.append(Paragraph(f"Chapter {i}: {title}", styles["Heading1"]))
        story.append(Spacer(1, 12))

        for p in content_elem.find_all("p"):
            text = p.get_text(strip=True)
            if text:
                story.append(Paragraph(text, styles["Normal"]))
                story.append(Spacer(1, 6))

        story.append(PageBreak())

    # Generar PDF
    doc.build(story)
    print(f"✅ Libro {nombre_pdf}.pdf generado correctamente en ../doc_pdf/")


In [23]:
# Realizamos el llamado a la funcion
link = "https://basecamp.com/gettingreal"
titulo= "Getting Real - Basecamp "
descripcion ="The smarter, faster, easier way to build a successful web application"

scrapping2pdf(link, titulo, descripcion)

✅ Libro gettingreal.pdf generado correctamente en ../doc_pdf/


In [24]:
# Realizamos el llamado a la funcion
link = "https://basecamp.com/shapeup"
titulo= "Shape Up - Basecamp "
descripcion ="Stop Running in Circles and Ship Work that Matters"

scrapping2pdf(link, titulo, descripcion)

✅ Libro shapeup.pdf generado correctamente en ../doc_pdf/
